### Pinecone Vector DB

In [1]:
import os

api_key = os.getenv("PINECONE_API_KEY")

In [ ]:
!pip install numpy==1.26.4 langchain langchain-pinecone langchain-huggingface

In [71]:
from pinecone import Pinecone
from langchain_huggingface import HuggingFaceEmbeddings

In [72]:
pc = Pinecone(api_key = api_key)


In [74]:
embeddings = HuggingFaceEmbeddings(model="intfloat/multilingual-e5-large", model_kwargs={'trust_remote_code': True})

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

In [75]:
sample_text = "This is a test sentence."
sample_embedding = embeddings.embed_query(sample_text)
embedding_dimension = len(sample_embedding)
print(f"The dimension of the HuggingFaceEmbeddings model is: {embedding_dimension}")

The dimension of the HuggingFaceEmbeddings model is: 1024


In [76]:
from pinecone import ServerlessSpec
index_name = "rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=embedding_dimension, # Use the correct dimension 384
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
index = pc.Index(index_name)
print(f"Index '{index_name}' recreated with dimension {embedding_dimension}.")
index


Index 'rag' recreated with dimension 1024.


In [77]:
from langchain_pinecone import PineconeVectorStore

In [78]:
vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings)

In [79]:
vector_store

In [80]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [81]:
vector_store.add_documents(documents)

['262bebf1-efa3-4796-83be-da579d76ba4b',
 'c41233ea-7cef-456d-8445-58d0b08df44d',
 'afbddf45-8b21-4301-8458-845fda8c9d5f',
 'aad305f6-70c5-4624-9ec6-cb035f30e6ab',
 'f9a267d3-2c23-4ca1-8ad4-9ca4a7a26ef6',
 'e2cf760a-dd3e-4b5e-8c7d-26141ea63bfb',
 '27219385-40b4-42b5-8187-fc556c8d881e',
 'cffdcc4b-2c33-4e81-9bdf-3a217cf7fdd2',
 'a4bde90e-1285-4296-b446-84cb1bbf6be4',
 '783ba393-36d5-4546-b959-1110c4b3ad3d']

In [82]:
### Search from Vector Store DB

vector_store.similarity_search("What is the weather")

[Document(id='c41233ea-7cef-456d-8445-58d0b08df44d', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='afbddf45-8b21-4301-8458-845fda8c9d5f', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='783ba393-36d5-4546-b959-1110c4b3ad3d', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='27219385-40b4-42b5-8187-fc556c8d881e', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]

First, we need to delete the existing Pinecone index.

In [83]:
if index_name in pc.list_indexes():
    pc.delete_index(index_name)
    print(f"Index '{index_name}' deleted.")
else:
    print(f"Index '{index_name}' does not exist.")

Index 'rag' does not exist.


Now, we will recreate the index with the correct dimension (384).

In [85]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "tweet"},
)
for res in results:
    print(f'* "{res.page_content}')

* "Building an exciting new project with LangChain - come check it out!
* "LangGraph is the best framework for building stateful, agentic applications!
* "I have a bad feeling I am going to get deleted :(
